# Does detector score track release date *within* the diffusion family?

`merge_scores.ipynb` §6 plotted per-generator median score against release date and fitted a
trend across **all** families at once. The markdown under that plot flagged why the slope was
not yet interpretable:

> this sample has 17 images per generator and confounds release date with family (GANs are
> early, diffusion is late), so a negative slope is not yet evidence of temporal decay on its own.

This notebook removes that confound the cheapest way available — hold the family fixed and look
at diffusion only — and replaces the eyeballed slope with a rank correlation and a p-value.

## What is being asked

Sixteen diffusion generators, released between 2020-06 and 2024-08. For each one, the median
score the panel gives its images. **Does that median move monotonically with release date?**

- **R** (Spearman's rho) — do the two quantities move together in *rank* order: does the
  newest generator get the lowest score, second-newest the second-lowest, and so on?
  `0` = no rank relationship, `-1` = perfectly reversed, `+1` = perfectly aligned. Rank-based,
  so it does not assume the decay is a straight line and one outlying generator cannot drive it.
- **p** — how often a correlation at least this strong appears when release dates are shuffled
  at random. Small p = hard to explain as coincidence.

**Why this matters for the thesis.** The framework dates a file by asking which generators
available by *T* could plausibly have produced it. Resolution *within* a family requires that
score carry some temporal signal once family is held fixed. If R is strong and significant here,
within-family temporal resolution is on the table. If R is ~0, the honest architecture treats a
family as one undifferentiated bucket and resolves time only *between* families.

## One decision that drives the whole result: the unit of analysis

**One row per generator (n = 16)**, not one row per image (n = 272).

All 17 images from Stable Diffusion 1.5 share a single release date. Treating them as 17
independent observations is pseudo-replication: it inflates n by 17× without adding any
temporal information, and shrinks the p-value by a factor that reflects nothing real. The
effective sample size for a question about *dates* is the number of distinct dates. §5
quantifies how badly that inflates things, because it is a trap worth being able to point at.

n = 16 is small, and the sensitivity check in §4 is not optional — it says which effect sizes
this test could have detected at all.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT   = Path.cwd()
SCORES = ROOT / "scores"

DETECTORS = ["cnndetection", "univfd",
             "dmimagedetection_progan", "dmimagedetection_latent",
             "aeroblade"]
zcols = {d: d.replace("dmimagedetection_", "dmid_") for d in DETECTORS}

REAL, FAKE = "#3b6ea5", "#c4462c"
NULLBAND, MUTED = "#d9d9d9", "#888888"

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 9, "axes.titlelocation": "left", "axes.titleweight": "bold",
})

N_PERM = 200_000   # p-value floor is 1/(N_PERM+1) ~ 5e-6
SEED   = 0

## 1. Rebuild the score table

Same join as `merge_scores.ipynb` §1, repeated here so this notebook stands alone rather than
depending on whether `scores/master_scores.csv` is current.

In [ ]:
man = pd.read_csv(ROOT / "manifest.csv")
M = man.copy()

for d in DETECTORS:
    s = pd.read_csv(SCORES / f"{d}.csv")
    assert list(s.columns) == ["image_id", "raw_score"], f"{d}: unexpected columns"
    M = M.merge(s.rename(columns={"raw_score": d}), on="image_id", how="left")

assert len(M) == len(man), "merge changed the row count"
for d in DETECTORS:
    assert M[d].notna().all(), f"{d}: {M[d].isna().sum()} images unscored"

M["release_date"] = pd.to_datetime(M.release_date, errors="coerce")

# sigma-above-authentic, as in merge_scores.ipynb section 4. This is a per-detector
# monotone linear rescaling, so it cannot change any rank correlation below -- section 2
# asserts that rather than asking you to take it on faith.
Z = M.copy()
for d in DETECTORS:
    ref = M.loc[M.label == 0, d]
    Z[d] = (M[d] - ref.mean()) / ref.std()

print(f"{len(M)} images, {len(DETECTORS)} detectors")

## 2. The diffusion-only table

Sixteen generators, 17 images each, one median score per detector.

In [ ]:
D = Z[(Z.label == 1) & (Z.generator_family == "diffusion")]

g = (D.groupby("generator", as_index=False)
      .agg(date=("release_date", "first"), n=("image_id", "size"),
           **{d: (d, "median") for d in DETECTORS})
      .sort_values("date").reset_index(drop=True))

assert g.n.nunique() == 1, "unbalanced images per generator"
assert g.date.is_monotonic_increasing and g.date.nunique() == len(g), "tied release dates"

x  = g.date.map(pd.Timestamp.toordinal).to_numpy(float)
nG = len(g)
print(f"n = {nG} diffusion generators, {g.n.iloc[0]} images each, "
      f"{g.date.min():%Y-%m} to {g.date.max():%Y-%m}")

g.assign(date=g.date.dt.strftime("%Y-%m-%d")).rename(columns=zcols).round(2)

## 3. Spearman R and a permutation p-value

Spearman's rho *is* Pearson's correlation computed on ranks, so `rankdata` + `np.corrcoef` is
the whole definition — using average ranks for ties makes it the tie-corrected form.

The p-value comes from **shuffling**, not from the usual `t = R·sqrt((n−2)/(1−R²))`
approximation. At n = 16 that approximation is doing real work on a small sample; shuffling the
release dates 200 000 times and counting how often chance alone produces a correlation this
strong makes no distributional assumption at all. It is also the honest way to report a floor:
a p-value from 200 000 shuffles cannot resolve below ~5e-6.

The null is built by permuting **each detector's own median-ranks**, not the integers 1…16.
Those coincide only when no two generators tie, and one pair does: Glide and Stable Diffusion 1.5
have byte-identical AEROBLADE medians. AEROBLADE's raw scores are stored at a precision coarse
relative to its dynamic range — 241 values recur across the 1224-image sample — so genuine ties
happen and the tie structure has to be carried into the null rather than assumed away.

Five detectors means five tests, so raw p-values are corrected with **Holm** — without it,
testing five things at α = 0.05 gives roughly a 1-in-4 chance that at least one comes up
"significant" by luck.

In [ ]:
def rankdata(a):
    """Average ranks; ties share their mean rank (scipy's 'average' method)."""
    a = np.asarray(a, float)
    r = np.empty(len(a), float)
    r[a.argsort()] = np.arange(1, len(a) + 1)
    for v in np.unique(a[pd.Series(a).duplicated(keep=False).to_numpy()]):
        m = a == v
        r[m] = r[m].mean()
    return r


def spearman_r(u, v):
    return float(np.corrcoef(rankdata(u), rankdata(v))[0, 1])


def perm_null(ranks_x, ranks_y, n_perm=N_PERM, seed=SEED):
    """Rho under H0: x-ranks fixed, y-ranks shuffled.

    Permuting the observed y-ranks rather than 1..n keeps any tie structure intact,
    which matters wherever two generators share a median.
    """
    P = np.argsort(np.random.default_rng(seed).random((n_perm, len(ranks_x))), axis=1)
    a = ranks_x - ranks_x.mean()
    B = ranks_y[P] - ranks_y.mean()
    return (B @ a) / np.sqrt((a @ a) * (B ** 2).sum(axis=1))


def perm_p(rho, null):
    """Two-sided, +1-corrected so a p-value is never reported as exactly zero."""
    return (np.sum(np.abs(null) >= abs(rho) - 1e-12) + 1) / (len(null) + 1)


def holm(pvals):
    p, m = np.asarray(pvals, float), len(pvals)
    adj, run = np.empty(m), 0.0
    for k, i in enumerate(p.argsort()):
        run = max(run, (m - k) * p[i])
        adj[i] = min(run, 1.0)
    return adj

In [ ]:
rx      = rankdata(x)
Draw    = M[(M.label == 1) & (M.generator_family == "diffusion")]
rows, nulls = [], {}

for d in DETECTORS:
    # the sigma rescaling is monotone, so raw and sigma scores must give identical rho
    raw = Draw.groupby("generator")[d].median().reindex(g.generator).to_numpy()
    assert abs(spearman_r(x, raw) - spearman_r(x, g[d])) < 1e-12, f"{d}: rho not scale-invariant"

    ry        = rankdata(g[d].to_numpy())
    nulls[d]  = perm_null(rx, ry)
    r         = spearman_r(x, g[d])
    rows.append({"detector": zcols[d], "R": r, "ties": int(g[d].duplicated().sum()),
                 "p_perm": perm_p(r, nulls[d])})

R = pd.DataFrame(rows)
R["p_holm"]  = holm(R.p_perm)
R["verdict"] = np.where(R.p_holm < 0.05, "significant", "not distinguishable from chance")

print(f"permutation null: {N_PERM:,} shuffles per detector, p floor {1/(N_PERM+1):.1e}\n")
print(R.assign(R=R.R.round(3),
               p_perm=R.p_perm.map("{:.2g}".format),
               p_holm=R.p_holm.map("{:.2g}".format)).to_string(index=False))

### Reading the two figures

**Top figure** — the data as it actually sits: median score against release date. **Bottom
figure** — the same 16 generators in *rank* space, which is literally where Spearman lives. If
R were +1 every point would land on the dashed diagonal; if −1, on the anti-diagonal. Rank space
is also where a single freak generator stops being able to swing the answer.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13.5, 5.4))

for j, d in enumerate(DETECTORS):
    r, p = R.R[j], R.p_perm[j]
    star = "*" if R.p_holm[j] < 0.05 else ""
    col  = FAKE if r < 0 else REAL
    ptxt = f"p<{1/(N_PERM+1):.0e}" if p <= 2/(N_PERM+1) else f"p={p:.3f}"

    ax = axes[0, j]
    ax.scatter(g.date, g[d], s=26, color=col, alpha=0.85, zorder=3,
               edgecolor="white", linewidth=0.6)
    ax.axhline(0, color=MUTED, lw=1, zorder=1)
    ax.set_title(f"{zcols[d]}\nR={r:+.2f}{star}  {ptxt}", fontsize=8)
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)

    ax = axes[1, j]
    ax.plot([1, nG], [1, nG], ls="--", lw=1, color=MUTED, zorder=1)
    ax.plot([1, nG], [nG, 1], ls=":", lw=1, color=MUTED, zorder=1)
    ax.scatter(rankdata(x), rankdata(g[d]), s=26, color=col, alpha=0.85, zorder=3,
               edgecolor="white", linewidth=0.6)
    ax.set_xlabel("rank of release date", fontsize=7.5)
    ax.set_xlim(0.2, nG + 0.8); ax.set_ylim(0.2, nG + 0.8)
    ax.tick_params(labelsize=7)

axes[0, 0].set_ylabel("median score\n(σ above authentic)")
axes[1, 0].set_ylabel("rank of median score")
fig.suptitle("diffusion family only (n=16 generators)   ·   red = score falls with release date"
             "   ·   * = significant after Holm",
             x=0.005, ha="left", fontsize=10, fontweight="bold")
fig.tight_layout()
plt.show()

## 4. Sensitivity: what could this test have detected?

A non-significant result at n = 16 is ambiguous — it could mean "no temporal signal" or "there
is one but 16 points cannot see it." Distinguishing those is not optional, and the permutation
null answers it directly: it *is* the distribution of R when release date carries no information,
so its 95th percentile is the smallest |R| this design can call significant.

Below that value, a real trend and no trend are indistinguishable here. Any detector whose R is
inside the gray band should be read as **"this experiment cannot tell,"** not as "no effect."

In [ ]:
crit = {d: float(np.quantile(np.abs(nulls[d]), 0.95)) for d in DETECTORS}
crit_holm = {d: float(np.quantile(np.abs(nulls[d]), 1 - 0.05 / len(DETECTORS)))
             for d in DETECTORS}
crit95 = max(crit.values())   # most conservative of the five, used for the band below

print(f"n = {nG}   ->   |R| must exceed ~{crit95:.3f} to reach p<0.05 uncorrected")
print(f"                           and ~{max(crit_holm.values()):.3f} to survive Holm "
      f"across {len(DETECTORS)} detectors")
print(f"\nper-detector 95th pct of |R| under H0 (they differ only via ties):")
for d in DETECTORS:
    print(f"  {zcols[d]:<24} {crit[d]:.3f}")

order = np.argsort(R.R.to_numpy())
fig, ax = plt.subplots(figsize=(7.4, 2.9))

ax.axvspan(-crit95, crit95, color=NULLBAND, zorder=0)
ax.axvline(0, color=MUTED, lw=1, zorder=1)

for row, j in enumerate(order):
    r, sig = R.R[j], R.p_holm[j] < 0.05
    col = FAKE if r < 0 else REAL
    ax.plot([0, r], [row, row], color=col, lw=2, zorder=2, solid_capstyle="round")
    ax.scatter([r], [row], s=95, zorder=3, color=col if sig else "white",
               edgecolor=col, linewidth=1.6)
    ax.text(r + (0.055 if r >= 0 else -0.055), row,
            f"{r:+.2f}" + ("  significant" if sig else "  n.s."),
            va="center", ha="left" if r >= 0 else "right", fontsize=7.5, color="#333")

ax.set_yticks(range(len(order)))
ax.set_yticklabels([R.detector[j] for j in order], fontsize=8)
ax.set_xlabel("Spearman R   (median score vs release date, diffusion only)")
ax.set_xlim(-1.28, 1.28)
ax.set_ylim(-0.6, len(order) - 0.4)
ax.grid(axis="y", visible=False)
ax.set_title(f"gray band = where chance alone reaches |R| 95% of the time at n={nG}"
             f"   ·   filled dot = survives Holm")
fig.tight_layout()
plt.show()

## 5. Why the per-image version is not the answer

The same test with one row per **image** instead of per generator. Nothing new is measured —
every image inherits its generator's date — but n goes from 16 to 272 and the p-values collapse.

This is the pseudo-replication trap, and it is the same failure mode CLAUDE.md flags for the
aggregator: *excess temporal granularity biases crossings earlier than warranted — the dangerous
direction*. A p-value manufactured by counting images rather than dates would make a temporal
claim look far better supported than the 16 generators behind it can support. Reported here only
so the inflation factor is on the record.

In [ ]:
xi = D.release_date.map(pd.Timestamp.toordinal).to_numpy(float)
cmp = pd.DataFrame({
    "detector":       [zcols[d] for d in DETECTORS],
    "R_per_gen":      R.R.round(3),
    "p_per_gen":      R.p_perm.map("{:.2g}".format),
    "R_per_image":    [round(spearman_r(xi, D[d]), 3) for d in DETECTORS],
})
rxi = rankdata(xi)                                  # heavily tied: 16 dates, 272 images
cmp["p_per_image"] = [
    perm_p(r, perm_null(rxi, rankdata(D[d].to_numpy()), n_perm=20_000))
    for r, d in zip(cmp.R_per_image, DETECTORS)
]
cmp["p_per_image"] = cmp.p_per_image.map("{:.2g}".format)

print(f"per-generator n = {nG}      per-image n = {len(D)}   "
      f"(but still only {nG} distinct dates)\n")
print(cmp.to_string(index=False))

## What this settles, and what it does not

| detector | R | p | Holm | |
|---|---|---|---|---|
| univfd | **−0.67** | 0.006 | **0.03** | significant |
| dmid_latent | +0.19 | 0.47 | 1 | inside the noise band |
| cnndetection | −0.17 | 0.54 | 1 | inside the noise band |
| dmid_progan | −0.14 | 0.61 | 1 | inside the noise band |
| aeroblade | −0.04 | 0.87 | 1 | inside the noise band |

**One detector out of five shows within-family temporal decay, and it is the one that was
supposed to be immune.** UniversalFakeDetect — the frozen-CLIP linear probe, the panel member
whose whole selling point is that a general-purpose backbone should generalise past its
training distribution — is the only one whose score falls monotonically with release date once
family is held fixed (R = −0.67, p = 0.006, still 0.03 after Holm). The confound in
`merge_scores.ipynb` §6 was real but it was not the *whole* story: for UnivFD there is a
genuine date effect underneath it.

**The other four results are "n = 16 cannot tell," not "no effect."** This is the part to not
get wrong. §4 shows |R| must exceed **0.50** to clear p < 0.05 at all, and **0.63** to survive
Holm across five detectors. CNNDetection's R = −0.17 is entirely consistent with a real but
moderate decay that 16 generators cannot resolve — and §5 of `merge_scores.ipynb` already
documents CNNDetection dropping sharply from GANs to diffusion, so a moderate within-family
trend is a live hypothesis this test simply lacks the power to reach. Absence of significance
here is a statement about the sample size, and nothing else.

**dmid_latent is the only member pointing the other way** (+0.19, n.s.), which is at least
directionally what its training data predicts: it is the latent-diffusion-trained checkpoint,
the one panel member whose training distribution moved *with* the family it is being tested on.
Not significant, so this is a hypothesis to test with more generators, not a finding.

**Consequence for the architecture.** The panel is not temporally homogeneous — one member
decays with release date and another may mildly improve. A max-over-generators aggregator built
on top of members that age at different rates inherits a date-dependent bias, so the score
models in the calibration layer cannot treat "detector" as exchangeable across the panel. That
is a constraint worth carrying forward regardless of what a larger sample does to the other four
numbers.

**What §5 costs you if you get it wrong.** The per-image version turns `dmid_latent` from
p = 0.47 into **p = 0.0066** — a null result rebranded as significant purely by counting 272
images instead of 16 dates. No new information entered; n was multiplied by 17. That single
number is the argument for keeping the generator as the unit of analysis everywhere in this
project.

Three caveats travel with every number here:

**17 images per generator.** Each median is a noisy estimate, and that noise attenuates R toward
zero — so a null result is weaker evidence against a trend than a significant result is for one.

**Release date is doing unexamined work, and the date distribution is lumpy.** CLAUDE.md flags
"generator availability date" as an unexamined primitive. Seven of the 16 generators fall in a
six-week window at the end of 2021, and the two most recent are both FLUX, released a day apart
— so the right-hand end of the UnivFD trend rests on effectively one generator-week. Sliding
Midjourney or the FLUX pair by a few months would move R, and this notebook does not test how
much. That sensitivity analysis is a prerequisite for putting R = −0.67 in the thesis.

**Date is still confounded with content and resolution inside the family.** Holding family fixed
removes the GAN/diffusion confound, not the fact that a 2020 DDPM sample and a 2024 FLUX sample
differ in subject matter and native resolution as well as in age. All five detectors see 200×200
crops (root README, Caveats), which bounds but does not eliminate the resolution part.